# Sumarização de Textos com PLN: Algoritmo de Luhn
**Autor: Wellington M Santos - Data Scientist**  
[![LinkedIn](https://img.shields.io/badge/LinkedIn-wellington--moreira--santos-blue)](https://www.linkedin.com/in/wellington-moreira-santos)
[![Email](https://img.shields.io/badge/Email-wsantos08%40hotmail.com-red)](mailto:wsantos08@hotmail.com)


---

## 1. Introdução

No projeto [Algoritmo Baseado em Frequência](Algoritmo%20baseado%20em%20frequencia.ipynb) implementei sumarização por frequência simples: conto palavras, normalizo pesos, somo a pontuação de cada sentença. Funciona, mas tem um ponto cego. Uma sentença pode acumular uma nota alta só porque contém várias palavras importantes espalhadas por ela inteira, mesmo que essas palavras estejam distantes umas das outras e sem relação direta de sentido.

Foi exatamente esse ponto cego que Hans Peter Luhn endereçou em 1958, num dos primeiros papers da história sobre sumarização automática. A ideia dele introduz um conceito novo: a proximidade. Palavras importantes que aparecem próximas umas das outras dentro de uma sentença formam um "cluster" significativo, e esse cluster é o que realmente indica que a sentença carrega conteúdo relevante. Uma sentença com termos importantes dispersos pontua menos do que uma sentença mais curta, mas com termos importantes concentrados.

Neste projeto, implemento o algoritmo de Luhn do zero, comparo com a abordagem por frequência simples, e expando o pipeline para um caso de uso mais completo: leitura de feeds RSS, geração de nuvem de palavras, extração de entidades nomeadas e exportação dos resumos em HTML.

**Referência teórica:** [Luhn, H.P. (1958). The Automatic Creation of Literature Abstracts](https://courses.ischool.berkeley.edu/i256/f06/papers/luhn58.pdf)

**Stack utilizada:** `Python 3.10+`, `nltk`, `collections`, `heapq`, `re`, `string`, `newspaper4k`, `feedparser`, `beautifulsoup4`, `wordcloud`, `matplotlib`, `spacy>=3.x`, `rouge-score`, `IPython.display`

**Seções:**

1. Introdução
2. Instalação e Configuração
3. Pré-processamento do Texto
4. A Lógica do Algoritmo de Luhn
5. Função de Sumarização
6. Visualização do Resumo
7. Extração de Texto da Web
8. Leitura de Feed RSS
9. Nuvem de Palavras
10. Extração de Entidades Nomeadas
11. Sumarização em Lote com Exportação HTML
12. Extensão: Lematização com spaCy
13. Avaliação com ROUGE: Luhn vs. Frequência Simples
14. Conclusão e Considerações Finais


---


## 2. Instalação e Configuração


In [1]:
# Instalar dependências
# !pip install nltk newspaper4k feedparser beautifulsoup4 wordcloud matplotlib spacy rouge-score
# !python -m spacy download pt_core_news_sm

In [2]:
import re
import os
import json
import string
import heapq
from collections import Counter

import nltk
from IPython.display import HTML, display

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

---

## 3. Pré-processamento do Texto

A normalização segue a mesma lógica do projeto de frequência simples: lowercase, tokenização, remoção de stopwords e pontuação. A diferença aqui é que vou adicionar duas stopwords customizadas ao vocabulário padrão, "ser" e "além", que no texto de exemplo aparecem com frequência mas não carregam carga temática relevante.


In [3]:
texto_original = """A inteligência artificial é a inteligência similar à humana máquinas.
                    Definem como o estudo de agente artificial com inteligência.
                    Ciência e engenharia de produzir máquinas com inteligência.
                    Resolver problemas e possuir inteligência.
                    Relacionada ao comportamento inteligente.
                    Construção de máquinas para raciocinar.
                    Aprender com os erros e acertos.
                    Inteligência artificial é raciocinar nas situações do cotidiano."""

texto_original = re.sub(r'\s+', ' ', texto_original)
print(texto_original)

A inteligência artificial é a inteligência similar à humana máquinas. Definem como o estudo de agente artificial com inteligência. Ciência e engenharia de produzir máquinas com inteligência. Resolver problemas e possuir inteligência. Relacionada ao comportamento inteligente. Construção de máquinas para raciocinar. Aprender com os erros e acertos. Inteligência artificial é raciocinar nas situações do cotidiano.


In [4]:
stopwords = nltk.corpus.stopwords.words('portuguese')
stopwords.extend(['ser', 'além'])
print(f"Total de stopwords: {len(stopwords)}")

Total de stopwords: 209


In [5]:
def preprocessamento(texto):
    """
    Normaliza o texto: lowercase, tokenização, remoção de stopwords e pontuação.

    Parâmetros:
        texto (str): texto bruto de entrada

    Retorna:
        str: string com tokens relevantes separados por espaço
    """
    texto_formatado = texto.lower()
    tokens = nltk.word_tokenize(texto_formatado, language='portuguese')
    tokens = [
        palavra for palavra in tokens
        if palavra not in stopwords and palavra not in string.punctuation
    ]
    return ' '.join([t for t in tokens if not t.isdigit()])

---

## 4. A Lógica do Algoritmo de Luhn

Aqui está a diferença central em relação ao projeto baseado em frequência. Em vez de somar diretamente os pesos das palavras importantes em cada sentença, o algoritmo de Luhn segue três passos:

1. Localizo, dentro de cada sentença, as posições onde aparecem as palavras consideradas importantes (as `top_n` mais frequentes do texto).
2. Agrupo essas posições em clusters: se a distância entre duas palavras importantes consecutivas for menor que um limiar definido, elas pertencem ao mesmo cluster.
3. Calculo a nota de cada cluster com a fórmula original de Luhn: o quadrado do número de palavras importantes no cluster, dividido pelo total de palavras que o cluster ocupa. A nota da sentença é a nota do melhor cluster encontrado nela.

Essa fórmula penaliza clusters "esticados": um cluster com 3 palavras importantes espalhadas em 10 posições pontua menos do que um cluster com as mesmas 3 palavras concentradas em 4 posições.

In [6]:
def calcula_nota_sentenca(sentencas, palavras_importantes, distancia):
    """
    Calcula a nota de cada sentença segundo o algoritmo de Luhn.

    Parâmetros:
        sentencas (list[str]): sentenças já pré-processadas
        palavras_importantes (list[str]): top N palavras mais frequentes do texto
        distancia (int): distância máxima entre posições para formar um cluster

    Retorna:
        list[tuple[float, int]]: pares (nota, índice_da_sentença)
    """
    notas = []

    for indice_sentenca, sentenca in enumerate(
        [nltk.word_tokenize(s.lower(), language='portuguese') for s in sentencas]
    ):
        indices_palavra = []
        for palavra in palavras_importantes:
            if palavra in sentenca:
                indices_palavra.append(sentenca.index(palavra))

        if not indices_palavra:
            continue

        indices_palavra.sort()

        # Agrupa índices próximos em clusters
        lista_grupos = []
        grupo = [indices_palavra[0]]
        for i in range(1, len(indices_palavra)):
            if indices_palavra[i] - indices_palavra[i - 1] < distancia:
                grupo.append(indices_palavra[i])
            else:
                lista_grupos.append(grupo)
                grupo = [indices_palavra[i]]
        lista_grupos.append(grupo)

        # Nota da sentença = nota do melhor cluster
        nota_maxima_grupo = 0
        for g in lista_grupos:
            palavras_importantes_no_grupo = len(g)
            total_palavras_no_grupo = g[-1] - g[0] + 1
            nota = 1.0 * palavras_importantes_no_grupo ** 2 / total_palavras_no_grupo
            nota_maxima_grupo = max(nota_maxima_grupo, nota)

        notas.append((nota_maxima_grupo, indice_sentenca))

    return notas


---


## 5. Função de Sumarização

A função `sumarizar` orquestra o pipeline inteiro: tokeniza sentenças, pré-processa cada uma, calcula a frequência das palavras, seleciona as `top_n` mais relevantes, e delega a pontuação para `calcula_nota_sentenca`.


In [7]:
def sumarizar(texto, top_n_palavras, distancia, quantidade_sentencas):
    """
    Executa o pipeline completo de sumarização pelo algoritmo de Luhn.

    Parâmetros:
        texto (str): texto original a ser sumarizado
        top_n_palavras (int): quantidade de palavras mais frequentes consideradas importantes
        distancia (int): distância máxima entre posições para formar um cluster
        quantidade_sentencas (int): número de sentenças no resumo

    Retorna:
        tuple: (sentencas_originais, melhores_sentencas, notas_sentencas)
    """
    sentencas_originais = nltk.sent_tokenize(texto, language='portuguese')
    sentencas_formatadas = [preprocessamento(s) for s in sentencas_originais]

    palavras = [
        palavra for sentenca in sentencas_formatadas
        for palavra in nltk.word_tokenize(sentenca)
    ]
    frequencia = Counter(palavras)
    top_palavras = [palavra for palavra, _ in frequencia.most_common(top_n_palavras)]

    notas_sentencas = calcula_nota_sentenca(sentencas_formatadas, top_palavras, distancia)
    melhores = heapq.nlargest(quantidade_sentencas, notas_sentencas)
    melhores_sentencas = [sentencas_originais[i] for (_, i) in melhores]

    return sentencas_originais, melhores_sentencas, notas_sentencas

In [8]:
sentencas_originais, melhores_sentencas, notas_sentencas = sumarizar(
    texto_original, top_n_palavras=5, distancia=3, quantidade_sentencas=3
)

print("Sentenças selecionadas:")
for s in melhores_sentencas:
    print(f"  - {s}")

print("\nNotas (nota, índice):")
print(notas_sentencas)

Sentenças selecionadas:
  - Inteligência artificial é raciocinar nas situações do cotidiano.
  - A inteligência artificial é a inteligência similar à humana máquinas.
  - Construção de máquinas para raciocinar.

Notas (nota, índice):
[(2.6666666666666665, 0), (2.0, 1), (2.0, 2), (1.0, 3), (2.0, 5), (3.0, 7)]



Três hiperparâmetros entram em jogo aqui: `top_n_palavras` define quantas palavras contam como "importantes", `distancia` define o quão próximas elas precisam estar para formar um cluster, e `quantidade_sentencas` define o tamanho do resumo final. Vale testar combinações diferentes para sentir o impacto de cada um.

---


## 6. Visualização do Resumo

Mantenho a mesma estratégia de visualização já utilizada anteriormente no projeto [baseado em frequências](Algoritmo%20baseado%20em%20frequencia.ipynb): detecto o ambiente de execução e ofereço HTML com destaque no Jupyter, ou texto puro como fallback em outros contextos.

In [10]:
def visualiza_resumo(titulo, lista_sentencas, melhores_sentencas):
    """
    Exibe o texto original com as sentenças do resumo destacadas.
    Suporta Jupyter (HTML) e outros ambientes (texto puro).

    Parâmetros:
        titulo (str): título exibido no cabeçalho
        lista_sentencas (list): todas as sentenças do texto original
        melhores_sentencas (list): sentenças selecionadas para o resumo
    """
    try:
        get_ipython  # noqa
        texto_html = ''
        for sentenca in lista_sentencas:
            if sentenca in melhores_sentencas:
                texto_html += f'<mark>{sentenca}</mark> '
            else:
                texto_html += sentenca + ' '
        display(HTML(f'<h3>Resumo: {titulo}</h3><p>{texto_html}</p>'))
    except NameError:
        print(f"\n=== Resumo: {titulo} ===")
        for sentenca in lista_sentencas:
            marcador = ">> " if sentenca in melhores_sentencas else "   "
            print(f"{marcador}{sentenca}")


In [11]:

visualiza_resumo('Texto de Exemplo', sentencas_originais, melhores_sentencas)


---



## 7. Extração de Texto da Web

In [12]:

# !pip install newspaper4k

In [13]:
from newspaper import Article

def extrair_artigo(url):
    """
    Extrai título e texto principal de uma URL usando newspaper4k.

    Parâmetros:
        url (str): endereço do artigo

    Retorna:
        tuple: (titulo, texto) ou (None, None) em caso de erro
    """
    try:
        artigo = Article(url, language='pt')
        artigo.download()
        artigo.parse()
        return artigo.title, artigo.text
    except Exception as e:
        print(f"Erro ao extrair {url}: {e}")
        return None, None

In [14]:
url = 'https://agenciabrasil.ebc.com.br/economia/noticia/2024-01/fmi-inteligencia-artificial-afetara-40-dos-empregos-em-todo-o-mundo'
titulo, texto = extrair_artigo(url)

print(f"Título: {titulo}")
print(f"Tamanho do texto: {len(texto)} caracteres")

Título: FMI: inteligência artificial afetará 40% dos empregos em todo o mundo
Tamanho do texto: 2124 caracteres


In [15]:
sentencas_originais, melhores_sentencas, notas_sentencas = sumarizar(
    texto, top_n_palavras=20, distancia=5, quantidade_sentencas=5
)
visualiza_resumo(titulo, sentencas_originais, melhores_sentencas)


Reparo que em textos reais, mais longos que o exemplo didático, preciso aumentar `top_n_palavras` e `distancia` proporcionalmente. Com `top_n_palavras=5` num artigo de centenas de palavras, praticamente nenhuma sentença teria termos importantes suficientes para formar um cluster relevante.



---



## 8. Leitura de Feed RSS

Aqui vou coletar um feed institucional estável e de fácil verificação: o feed da Agência Brasil.


In [16]:

# !pip install feedparser beautifulsoup4


In [18]:
import feedparser
from bs4 import BeautifulSoup

# url_feed = 'https://agenciabrasil.ebc.com.br/feed/'
url_feed = 'http://agenciabrasil.ebc.com.br/rss/ultimasnoticias/feed.xml'
feed = feedparser.parse(url_feed)

print(f"Total de entradas no feed: {len(feed.entries)}")
print(f"Título do feed: {feed.feed.get('title', 'N/A')}")

Total de entradas no feed: 10
Título do feed: Feed Últimas


In [19]:
for entrada in feed.entries[:3]:
    print(entrada.title)
    print(entrada.link)
    print('---')

Sistema de notificação de desastre evoluiu, mas ainda tem fragilidades
https://agenciabrasil.ebc.com.br/geral/noticia/2026-06/sistema-de-notificacao-de-desastre-evoluiu-mas-ainda-tem-fragilidades
---
Holanda atropela Suécia por 5 a 1 e mostra força no grupo F
https://agenciabrasil.ebc.com.br/esportes/noticia/2026-06/holanda-atropela-suecia-por-5-1-e-mostra-forca-no-grupo-f
---
LBF: TV Brasil transmite Sodiê Mesquita x Unimed Campinas no domingo
https://agenciabrasil.ebc.com.br/esportes/noticia/2026-06/lbf-tv-brasil-transmite-sodie-mesquita-x-unimed-campinas-no-domingo
---



O conteúdo de cada entrada vem com tags HTML embutidas. Preciso limpar isso antes de alimentar o pipeline de sumarização.


In [20]:
def limpa_html(texto):
    """
    Remove tags HTML de um trecho de texto, retornando apenas o conteúdo textual.

    Parâmetros:
        texto (str): texto com marcação HTML

    Retorna:
        str: texto limpo
    """
    if not texto:
        return ''
    return BeautifulSoup(texto, 'html.parser').get_text()

In [21]:
artigos_feed = []
for entrada in feed.entries:
    conteudo_bruto = entrada.get('summary', '') or entrada.get('description', '')
    artigos_feed.append({
        'titulo': entrada.title,
        'link': entrada.link,
        'conteudo': limpa_html(conteudo_bruto)
    })

print(f"Artigos processados: {len(artigos_feed)}")
print(artigos_feed[0])

Artigos processados: 10
{'titulo': 'Sistema de notificação de desastre evoluiu, mas ainda tem fragilidades', 'link': 'https://agenciabrasil.ebc.com.br/geral/noticia/2026-06/sistema-de-notificacao-de-desastre-evoluiu-mas-ainda-tem-fragilidades', 'conteudo': '\n\nNa madrugada deste sábado (20), uma invasão ao sistema Defesa Civil Alerta chamou a atenção para a fragilidade na segurança de uma das principais ferramentas de proteção da população em casos de desastres naturais, ao transmitir uma mensagem de Alerta Extremo falsa para milhões de aparelhos celulares em várias regiões do país.\xa0\nA falha foi reconhecida pelo secretário Nacional de Proteção e Defesa Civil do Ministério da Integração e do Desenvolvimento Regional, Wolnei Wolff, em entrevista à imprensa.\nNotícias relacionadas:Falso alerta da Defesa Civil atingiu cerca de 30 milhões em 8 estados.Defesa Civil: 10 alertas falsos foram disparados em invasão de sistema.“Já se encontra em desenvolvimento dentro do Ministério da Integr


Uso `html.parser`, nativo do `BeautifulSoup`, em vez de `html5lib`. Evita uma dependência extra e é suficiente para limpar marcação simples de feeds RSS.


In [22]:
# Persistindo os artigos em disco para reuso
with open('feed_agencia_brasil.json', 'w', encoding='utf-8') as arquivo:
    json.dump(artigos_feed, arquivo, indent=2, ensure_ascii=False)

In [23]:
with open('feed_agencia_brasil.json', encoding='utf-8') as arquivo:
    artigos_feed = json.load(arquivo)

In [24]:
artigos_feed

[{'titulo': 'Sistema de notificação de desastre evoluiu, mas ainda tem fragilidades',
  'link': 'https://agenciabrasil.ebc.com.br/geral/noticia/2026-06/sistema-de-notificacao-de-desastre-evoluiu-mas-ainda-tem-fragilidades',
  'conteudo': '\n\nNa madrugada deste sábado (20), uma invasão ao sistema Defesa Civil Alerta chamou a atenção para a fragilidade na segurança de uma das principais ferramentas de proteção da população em casos de desastres naturais, ao transmitir uma mensagem de Alerta Extremo falsa para milhões de aparelhos celulares em várias regiões do país.\xa0\nA falha foi reconhecida pelo secretário Nacional de Proteção e Defesa Civil do Ministério da Integração e do Desenvolvimento Regional, Wolnei Wolff, em entrevista à imprensa.\nNotícias relacionadas:Falso alerta da Defesa Civil atingiu cerca de 30 milhões em 8 estados.Defesa Civil: 10 alertas falsos foram disparados em invasão de sistema.“Já se encontra em desenvolvimento dentro do Ministério da Integração, dentro da nos

---